Turning Gaming Cafes into streaming and gaming hubs.

In [61]:
# Imports
import os
import io
import json
import time
import random
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv
import requests
from azure.storage.blob import BlobServiceClient

In [62]:
# 1. Setup environment and directories
BASE_DIR   = Path.cwd().resolve().parent
CONFIG_DIR = BASE_DIR / "config"

with open(CONFIG_DIR / "project_settings.json", "r", encoding="utf-8") as f:
    SETTINGS = json.load(f)

RAW_TWITCH_DIR = BASE_DIR / "data" / "raw" / "twitch"
RAW_STEAM_DIR  = BASE_DIR / "data" / "raw" / "steam"
RAW_TWITCH_DIR.mkdir(parents=True, exist_ok=True)
RAW_STEAM_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR   :", BASE_DIR)
print("Project    :", SETTINGS["project_name"])
print("Target Games:", SETTINGS["target_games"])

BASE_DIR   : D:\Msc_Data_Analytics\Data_Intensive_Scalable_System\CA2
Project    : hybrid_streaming_play_hubs
Target Games: ['Counter-Strike', 'Apex Legends', 'Fortnite', 'Dota 2', "Tom Clancy's Rainbow Six Siege"]


In [63]:
# 2. Load environment variables and initialise Azure Blob client
ENV_PATH = CONFIG_DIR / ".env"
load_dotenv(dotenv_path=ENV_PATH, override=True)

CONN_STR = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
blob_service = BlobServiceClient.from_connection_string(CONN_STR)

def upload_folder_to_blob(local_folder: Path, container_name: str):
    """Upload all files in a local folder to an Azure Blob container."""
    container_client = blob_service.get_container_client(container_name)
    for file_path in local_folder.iterdir():
        if file_path.is_file():
            with open(file_path, "rb") as f:
                container_client.upload_blob(name=file_path.name, data=f, overwrite=True)
            print(f"  Uploaded  {container_name}/{file_path.name}")

print("Azure Blob client ready.")

Azure Blob client ready.


In [64]:
# 3. Authenticate with Twitch API
client_id     = os.getenv("TWITCH_CLIENT_ID")
client_secret = os.getenv("TWITCH_CLIENT_SECRET")

def generate_twitch_token(client_id, client_secret):
    auth_url    = "https://id.twitch.tv/oauth2/token"
    auth_params = {
        "client_id":     client_id,
        "client_secret": client_secret,
        "grant_type":    "client_credentials"
    }
    resp = requests.post(auth_url, params=auth_params, timeout=15)
    data = resp.json()
    print("Access Token Generated:", data["access_token"])
    return data["access_token"]

if client_id and client_secret:
    access_token = generate_twitch_token(client_id, client_secret)
else:
    print("Twitch credentials missing in .env")

Access Token Generated: 44yw7aik1a5qebpjw2j4b0qwi7u1rg


In [65]:
# 4. Retrieve Twitch game IDs
timestamp    = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")
target_games = SETTINGS["target_games"]

headers = {
    "Client-ID":     client_id,
    "Authorization": f"Bearer {access_token}"
}

game_id_map = {}
games_url   = "https://api.twitch.tv/helix/games"

for game in target_games:
    resp = requests.get(games_url, headers=headers, params={"name": game}, timeout=15)
    data = resp.json()
    if "data" in data and data["data"]:
        game_id_map[game] = data["data"][0]["id"]

print("Game ID Map:")
print(game_id_map)

with open(RAW_TWITCH_DIR / f"game_ids_{timestamp}.json", "w", encoding="utf-8") as f:
    json.dump(game_id_map, f, indent=2)
print("Game IDs saved.")

Game ID Map:
{'Counter-Strike': '32399', 'Apex Legends': '511224', 'Fortnite': '33214', 'Dota 2': '29595', "Tom Clancy's Rainbow Six Siege": '460630'}
Game IDs saved.


In [66]:
# 5. Download Twitch streams for each game
all_stream_data = {}
streams_url     = "https://api.twitch.tv/helix/streams"

for game_name, game_id in game_id_map.items():
    resp = requests.get(
        streams_url,
        headers=headers,
        params={"game_id": game_id, "first": 100},
        timeout=15
    )
    data = resp.json()
    all_stream_data[game_name] = data

    file_name = f"streams_{game_name.lower().replace(' ', '_').replace('-', '_')}_{timestamp}.json"
    with open(RAW_TWITCH_DIR / file_name, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    print(f"{game_name}: {len(data.get('data', []))} streams saved")

print("All Twitch stream JSON files saved.")

Counter-Strike: 96 streams saved
Apex Legends: 100 streams saved
Fortnite: 97 streams saved
Dota 2: 98 streams saved
Tom Clancy's Rainbow Six Siege: 100 streams saved
All Twitch stream JSON files saved.


In [ ]:
# 6. Flatten Twitch stream data into a CSV
flat_rows = []
for game_name, response_data in all_stream_data.items():
    for stream in response_data.get("data", []):
        flat_rows.append({
            "snapshot_time_utc": timestamp,
            "game_name":         game_name,
            "stream_id":         stream.get("id"),
            "user_id":           stream.get("user_id"),
            "user_login":        stream.get("user_login"),
            "user_name":         stream.get("user_name"),
            "title":             stream.get("title"),
            "viewer_count":      stream.get("viewer_count"),
            "started_at":        stream.get("started_at"),
            "language":          stream.get("language"),
            "thumbnail_url":     stream.get("thumbnail_url"),
            "is_mature":         stream.get("is_mature"),
        })

twitch_flat_df = pd.DataFrame(flat_rows)
csv_path = RAW_TWITCH_DIR / f"twitch_streams_flat_{timestamp}.csv"
twitch_flat_df.to_csv(csv_path, index=False)

print(f"Flat CSV saved: {csv_path.name}")
print(f"Total stream rows: {len(twitch_flat_df)}")

twitch_flat_df.head(5)

Flat CSV saved: twitch_streams_flat_2026-05-01_181607.csv
Total stream rows: 491


,snapshot_time_utc,game_name,stream_id,user_id,user_login,user_name,title,viewer_count,started_at,language,thumbnail_url,is_mature
0,2026-05-01_181607,Counter-Strike,317024127832,14371185,northernlion,Northernlion,it's the first of the month...so cache your du...,11983,2026-05-01T16:00:25Z,en,https://static-cdn.jtvnw.net/previews-ttv/live...,False
1,2026-05-01_181607,Counter-Strike,315961115222,103577259,jlcs2,jLcs2,🔴 4000 ELO FACEIT PLAYGROUND w/ lil pup @magil...,6157,2026-05-01T15:42:07Z,en,https://static-cdn.jtvnw.net/previews-ttv/live...,True
2,2026-05-01_181607,Counter-Strike,315931968597,39154778,recrent,Recrent,ВОЗВРАЩАЮ 3000 ELO ПРИНАДЛЕЖАЩИЕ МНЕ ПО ПРАВУ ...,5156,2026-05-01T13:15:35Z,ru,https://static-cdn.jtvnw.net/previews-ttv/live...,False
3,2026-05-01_181607,Counter-Strike,319058672986,1370325422,nocriesof,nocriesof,FACEIT TOP 40 NA W/ @jasonr | !tg,2988,2026-05-01T14:58:38Z,en,https://static-cdn.jtvnw.net/previews-ttv/live...,False
4,2026-05-01_181607,Counter-Strike,316991395431,86131599,lol_nemesis,lol_nemesis,Variety time !korea !patreon !nord !discord,2775,2026-05-01T12:17:56Z,en,https://static-cdn.jtvnw.net/previews-ttv/live...,False


In [68]:
# 7. Upload Twitch raw data to Azure Blob Storage
print("Uploading Twitch raw data to Azure Blob (raw-twitch)...")
upload_folder_to_blob(RAW_TWITCH_DIR, "raw-twitch")
print("\nTwitch ingestion complete.")

Uploading Twitch raw data to Azure Blob (raw-twitch)...
  Uploaded  raw-twitch/game_ids_2026-05-01_015443.json
  Uploaded  raw-twitch/game_ids_2026-05-01_140409.json
  Uploaded  raw-twitch/game_ids_2026-05-01_140555.json
  Uploaded  raw-twitch/game_ids_2026-05-01_140827.json
  Uploaded  raw-twitch/game_ids_2026-05-01_141402.json
  Uploaded  raw-twitch/game_ids_2026-05-01_143748.json
  Uploaded  raw-twitch/game_ids_2026-05-01_144703.json
  Uploaded  raw-twitch/game_ids_2026-05-01_150115.json
  Uploaded  raw-twitch/game_ids_2026-05-01_151232.json
  Uploaded  raw-twitch/game_ids_2026-05-01_181607.json
  Uploaded  raw-twitch/streams_apex_legends_2026-05-01_015443.json
  Uploaded  raw-twitch/streams_apex_legends_2026-05-01_140409.json
  Uploaded  raw-twitch/streams_apex_legends_2026-05-01_140555.json
  Uploaded  raw-twitch/streams_apex_legends_2026-05-01_140827.json
  Uploaded  raw-twitch/streams_apex_legends_2026-05-01_141402.json
  Uploaded  raw-twitch/streams_apex_legends_2026-05-01_1437

In [69]:
# 8. Retrieve Steam reviews for priority games
PRIORITY_GAMES = {
    730:     "Counter-Strike",
    1172470: "Apex Legends",
    1172620: "Fortnite",
    570:     "Dota 2",
    359550:  "Tom Clancy's Rainbow Six Siege",
}

MAX_REVIEWS_PER_GAME = 200   # 200 × 5 = 1000 reviews max — manageable & representative
MIN_REVIEW_LEN       = 30    # Filter out very short noise reviews
SLEEP_SECONDS        = 0.5   # Polite rate limiting

print("Steam priority games:")
for appid, name in PRIORITY_GAMES.items():
    print(f"  {appid}: {name}")

Steam priority games:
  730: Counter-Strike
  1172470: Apex Legends
  1172620: Fortnite
  570: Dota 2
  359550: Tom Clancy's Rainbow Six Siege


In [70]:
# 9. Fetch Steam reviews for each priority game
all_reviews = []

for appid, game_name in PRIORITY_GAMES.items():
    print(f"\nFetching reviews : {game_name} (appid={appid})")

    params = {
        "json":                     1,
        "language":                 "english",
        "filter":                   "recent",
        "review_type":              "all",
        "purchase_type":            "all",
        "num_per_page":             100,
        "filter_offtopic_activity": 0,
    }

    fetched = 0
    cursor  = "*"

    while fetched < MAX_REVIEWS_PER_GAME:
        params["cursor"] = cursor
        try:
            resp = requests.get(
                f"https://store.steampowered.com/appreviews/{appid}",
                params=params,
                timeout=30
            )
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"  Request failed: {e}")
            break

        reviews = data.get("reviews", [])
        if not reviews:
            break

        for r in reviews:
            review_text = r.get("review", "")
            if len(review_text) < MIN_REVIEW_LEN:
                continue
            all_reviews.append({
                "appid":              appid,
                "game_name":          game_name,
                "review_id":          r.get("recommendationid"),
                "voted_up":           r.get("voted_up"),
                "votes_up":           r.get("votes_up", 0),
                "playtime_at_review": r.get("author", {}).get("playtime_at_review", 0),
                "review_text":        review_text[:500],
                "timestamp_created":  r.get("timestamp_created"),
                "language":           r.get("language"),
            })
            fetched += 1
            if fetched >= MAX_REVIEWS_PER_GAME:
                break

        new_cursor = data.get("cursor", "")
        if not new_cursor or new_cursor == cursor:
            break
        cursor = new_cursor
        time.sleep(SLEEP_SECONDS)

    print(f"  {fetched} reviews collected")

print(f"\nTotal reviews collected: {len(all_reviews)}")


Fetching reviews : Counter-Strike (appid=730)
  200 reviews collected

Fetching reviews : Apex Legends (appid=1172470)
  200 reviews collected

Fetching reviews : Fortnite (appid=1172620)
  200 reviews collected

Fetching reviews : Dota 2 (appid=570)
  200 reviews collected

Fetching reviews : Tom Clancy's Rainbow Six Siege (appid=359550)
  200 reviews collected

Total reviews collected: 1000


In [71]:
# 10. Process Steam reviews and compute sentiment metrics
steam_reviews_df = pd.DataFrame(all_reviews)

if not steam_reviews_df.empty:
    sentiment = (
        steam_reviews_df
        .groupby(["appid", "game_name"])
        .agg(
            review_count   = ("review_id",           "count"),
            positive_count = ("voted_up",             "sum"),
            avg_playtime   = ("playtime_at_review",   "mean")
        )
        .reset_index()
    )
    sentiment["sentiment_score"]  = (sentiment["positive_count"] / sentiment["review_count"]).round(4)
    sentiment["review_velocity"]  = (sentiment["review_count"] / 7).round(1)
    sentiment["avg_playtime_hrs"] = (sentiment["avg_playtime"] / 60).round(1)
else:
    print("No reviews fetched — using documented fallback values")
    sentiment = pd.DataFrame([
        {"appid": 730,    "game_name": "Counter-Strike",    "review_count": 200, "positive_count": 148, "sentiment_score": 0.74, "review_velocity": 28.6, "avg_playtime_hrs": 85.0},
        {"appid": 1172470,"game_name": "Apex Legends",      "review_count": 200, "positive_count": 142, "sentiment_score": 0.71, "review_velocity": 28.6, "avg_playtime_hrs": 38.0},
        {"appid": 1172620,"game_name": "Fortnite",          "review_count": 200, "positive_count": 132, "sentiment_score": 0.66, "review_velocity": 28.6, "avg_playtime_hrs": 25.0},
        {"appid": 570,    "game_name": "Dota 2",            "review_count": 200, "positive_count": 162, "sentiment_score": 0.81, "review_velocity": 28.6, "avg_play_time_hrs": 120.0},
        {"appid": 359550, "game_name": "Rainbow Six Siege", "review_count": 200, "positive_count": 144, "sentiment_score": 0.72, "review_velocity": 28.6, "avg_play_time_hrs": 65.0},
    ])

print("\nSentiment summary per game:")
print(sentiment[["game_name", "review_count", "sentiment_score", "review_velocity"]].to_string(index=False))

raw_reviews_path = RAW_STEAM_DIR / f"steam_reviews_raw_{timestamp}.csv"
sentiment_path   = RAW_STEAM_DIR / "steam_sentiment_summary.csv"

steam_reviews_df.to_csv(raw_reviews_path, index=False)
sentiment.to_csv(sentiment_path, index=False)

print(f"\n Raw reviews saved:       {raw_reviews_path.name}")
print(f"Sentiment summary saved: {sentiment_path.name}")


Sentiment summary per game:
                     game_name  review_count  sentiment_score  review_velocity
                        Dota 2           200            0.700             28.6
                Counter-Strike           200            0.725             28.6
Tom Clancy's Rainbow Six Siege           200            0.685             28.6
                  Apex Legends           200            0.595             28.6
                      Fortnite           200            0.660             28.6

 Raw reviews saved:       steam_reviews_raw_2026-05-01_181607.csv
Sentiment summary saved: steam_sentiment_summary.csv


In [72]:
# 11. Save Games-Stats hype data
games_stats_data = [
    {"game_name": "Counter-Strike",                 "appid": 730,    "followers": 1_900_000, "follower_growth":  500, "pre_release_flag": False},
    {"game_name": "Apex Legends",                   "appid": 1172470,"followers": 1_200_000, "follower_growth":  350, "pre_release_flag": False},
    {"game_name": "Fortnite",                       "appid": 1172620,"followers": 2_100_000, "follower_growth": 1100, "pre_release_flag": True},
    {"game_name": "Dota 2",                         "appid": 570,    "followers": 3_200_000, "follower_growth":  700, "pre_release_flag": False},
    {"game_name": "Tom Clancy's Rainbow Six Siege", "appid": 359550, "followers": 1_500_000, "follower_growth":  420, "pre_release_flag": False},
]

hype_path = RAW_STEAM_DIR / "games_stats_hype.json"
with open(hype_path, "w", encoding="utf-8") as f:
    json.dump(games_stats_data, f, indent=2)

print("Games-Stats hype data saved:")
for g in games_stats_data:
    print(f"  {g['game_name']}: {g['followers']:,} followers | growth: {g['follower_growth']}/day | pre-release: {g['pre_release_flag']}")

Games-Stats hype data saved:
  Counter-Strike: 1,900,000 followers | growth: 500/day | pre-release: False
  Apex Legends: 1,200,000 followers | growth: 350/day | pre-release: False
  Fortnite: 2,100,000 followers | growth: 1100/day | pre-release: True
  Dota 2: 3,200,000 followers | growth: 700/day | pre-release: False
  Tom Clancy's Rainbow Six Siege: 1,500,000 followers | growth: 420/day | pre-release: False


In [73]:
# 12. Upload Steam and Games-Stats data to Azure Blob Storage
print("Uploading Steam + Games-Stats data to Azure Blob (raw-steam)...")
upload_folder_to_blob(RAW_STEAM_DIR, "raw-steam")
print("\nSteam ingestion complete.")

Uploading Steam + Games-Stats data to Azure Blob (raw-steam)...
  Uploaded  raw-steam/games_filtered.csv
  Uploaded  raw-steam/games_stats_hype.json
  Uploaded  raw-steam/steam_reviews_raw_2026-05-01_144703.csv
  Uploaded  raw-steam/steam_reviews_raw_2026-05-01_150115.csv
  Uploaded  raw-steam/steam_reviews_raw_2026-05-01_151232.csv
  Uploaded  raw-steam/steam_reviews_raw_2026-05-01_181607.csv
  Uploaded  raw-steam/steam_sentiment_summary.csv

Steam ingestion complete.


In [74]:
# 13. Create and upload Gaming Café dataset
cafes_ireland = [
    {"cafe_id": 1, "name": "Pixel Palace",        "city": "Dublin",    "country": "Ireland", "latitude": 53.3498, "longitude": -6.2603, "capacity": 40, "features": "Streaming, LAN"},
    {"cafe_id": 2, "name": "LevelUp Lounge",       "city": "Dublin",    "country": "Ireland", "latitude": 53.3401, "longitude": -6.2611, "capacity": 30, "features": "LAN, Snack Bar"},
    {"cafe_id": 3, "name": "GameZone Cork",         "city": "Cork",      "country": "Ireland", "latitude": 51.8985, "longitude": -8.4756, "capacity": 25, "features": "Streaming"},
    {"cafe_id": 4, "name": "EsportsHub Galway",     "city": "Galway",    "country": "Ireland", "latitude": 53.2743, "longitude": -9.0514, "capacity": 20, "features": "Tournaments"},
    {"cafe_id": 5, "name": "Arena Limerick",        "city": "Limerick",  "country": "Ireland", "latitude": 52.6680, "longitude": -8.6305, "capacity": 35, "features": "LAN, Streaming"},
    {"cafe_id": 6, "name": "Nexus Gaming",          "city": "Belfast",   "country": "Ireland", "latitude": 54.5973, "longitude": -5.9301, "capacity": 50, "features": "VR, Streaming"},
    {"cafe_id": 7, "name": "RetroPlay Waterford",   "city": "Waterford", "country": "Ireland", "latitude": 52.2593, "longitude": -7.1101, "capacity": 15, "features": "LAN"},
    {"cafe_id": 8, "name": "ProGamer Kilkenny",     "city": "Kilkenny",  "country": "Ireland", "latitude": 52.6541, "longitude": -7.2448, "capacity": 20, "features": "Tournaments, LAN"},
]

RAW_CAFES_DIR = BASE_DIR / "data" / "raw" / "cafes"
RAW_CAFES_DIR.mkdir(parents=True, exist_ok=True)

cafes_path = RAW_CAFES_DIR / "cafes_ireland.json"
with open(cafes_path, "w", encoding="utf-8") as f:
    json.dump(cafes_ireland, f, indent=2)

print(f"Cafe data saved locally: {cafes_path.name} ({len(cafes_ireland)} venues)")

# Auto-create container if it doesn't exist, then upload
CAFE_CONTAINER = "raw-cafes"
container_client = blob_service.get_container_client(CAFE_CONTAINER)

if not container_client.exists():
    container_client.create_container()
    print(f"  Container '{CAFE_CONTAINER}' created")
else:
    print(f"  ! Container '{CAFE_CONTAINER}' already exists")

with open(cafes_path, "rb") as f:
    container_client.upload_blob(name=cafes_path.name, data=f, overwrite=True)

print(f"  Uploaded: {CAFE_CONTAINER}/{cafes_path.name}")

Cafe data saved locally: cafes_ireland.json (8 venues)
  ! Container 'raw-cafes' already exists
  Uploaded: raw-cafes/cafes_ireland.json
